In [3]:
import os
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix

# Set clean visualization themes
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("🚀 Step 1 & 2: Loading & Cleaning Dataset...")
# Load dataset
df = pd.read_csv('online_retail.csv')

# Remove rows missing critical identification info
df = df.dropna(subset=['CustomerID', 'Description'])
df['CustomerID'] = df['CustomerID'].astype(int)

# Exclude cancelled invoices (InvoiceNo starting with 'C')
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

# Filter out non-positive quantities and prices
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]

# Standarize Date formats and extract transaction total amount
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['TotalAmount'] = df['Quantity'] * df['UnitPrice']

print(f"Dataset preprocessed successfully. Clean data shape: {df.shape}")


print("\n📊 Step 3: Fast-Track Exploratory Data Analysis Metrics...")
# Top 5 items sold
top_products = df.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(5)
print("Top 5 Selling Products by Volume:\n", top_products)

# Top 5 market locations
country_counts = df['Country'].value_counts().head(5)
print("\nTop 5 Countries by Transactions:\n", country_counts)


print("\n📈 Step 4: RFM Feature Engineering...")
# Benchmark date set 1 day after the latest recorded transaction
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

# Group transactions securely per individual buyer
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalAmount': 'sum'
}).reset_index()

# Rename columns to standard framework metrics
rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

# Apply natural log transform log1p to suppress extreme data skews
X = rfm[['Recency', 'Frequency', 'Monetary']]
X_log = np.log1p(X)

# Normalize the scores using standard scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_log)


print("\n🧮 Step 5: Clustering Methodology & Evaluation...")
# Run silhouette checking loops to confirm optimal metrics
k_range = range(2, 6)
for k in k_range:
    test_km = KMeans(n_clusters=k, random_state=42, n_init=10)
    test_labels = test_km.fit_predict(X_scaled)
    s_score = silhouette_score(X_scaled, test_labels)
    print(f"Clusters k={k} | Silhouette Score: {s_score:.4f} | Inertia: {test_km.inertia_:.2f}")

# Train the optimal cluster framework
optimal_k = 4
best_kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)

# FIXED: fit_predict yields a safe 1D array profile mapping directly to the rows
rfm['Cluster'] = best_kmeans.fit_predict(X_scaled)

# DYNAMIC INTERPRETATION: Identify centers mathematically to attach proper profile names
cluster_means = rfm.groupby('Cluster')[['Recency', 'Frequency', 'Monetary']].mean()
print("\nDiscovered Mathematical Cluster Means:\n", cluster_means)

cluster_mapping = {}
for cluster_idx, row in cluster_means.iterrows():
    if row['Monetary'] == cluster_means['Monetary'].max():
        cluster_mapping[cluster_idx] = 'High-Value'
    elif row['Recency'] == cluster_means['Recency'].max():
        cluster_mapping[cluster_idx] = 'At-Risk'
    elif row['Frequency'] >= cluster_means['Frequency'].median():
        cluster_mapping[cluster_idx] = 'Regular'
    else:
        cluster_mapping[cluster_idx] = 'Occasional'

rfm['Segment'] = rfm['Cluster'].map(cluster_mapping)
print("\nAssigned Demographic Segment Counts:\n", rfm['Segment'].value_counts())


print("\n🛒 Step 6: Memory-Safe Sparse Collaborative Filtering Matrix Construction...")
# Aggregate orders per item/user pair to downsize layout allocations
item_user_counts = df.groupby(['Description', 'CustomerID'])['Quantity'].sum().reset_index()

# Group tokens structurally into categories
item_user_counts['Description'] = item_user_counts['Description'].astype('category')
item_user_counts['CustomerID'] = item_user_counts['CustomerID'].astype('category')

# Construct sparse compressed matrix to avoid heavy dense arrays
sparse_matrix = csr_matrix((
    item_user_counts['Quantity'], 
    (item_user_counts['Description'].cat.codes, item_user_counts['CustomerID'].cat.codes)
))

# Generate user-item recommendation similarity indices
similarity_matrix = cosine_similarity(sparse_matrix, dense_output=True)
unique_products = item_user_counts['Description'].cat.categories

item_similarity_df = pd.DataFrame(
    similarity_matrix, 
    index=unique_products, 
    columns=unique_products
)
print(f"Product Similarity Lookup Array created. Shape: {item_similarity_df.shape}")


print("\n💾 Step 7: Exporting Assets to Deployment Folder...")
os.makedirs('models', exist_ok=True)

# Export assets safely via pickle serialization
with open('models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open('models/kmeans.pkl', 'wb') as f:
    pickle.dump(best_kmeans, f)
with open('models/cluster_mapping.pkl', 'wb') as f:
    pickle.dump(cluster_mapping, f)
with open('models/item_similarity_df.pkl', 'wb') as f:
    pickle.dump(item_similarity_df, f)

print("🎉 Execution success! All pipeline elements saved inside 'models/' without faults.")

🚀 Step 1 & 2: Loading & Cleaning Dataset...
Dataset preprocessed successfully. Clean data shape: (397884, 9)

📊 Step 3: Fast-Track Exploratory Data Analysis Metrics...
Top 5 Selling Products by Volume:
 Description
PAPER CRAFT , LITTLE BIRDIE           80995
MEDIUM CERAMIC TOP STORAGE JAR        77916
WORLD WAR 2 GLIDERS ASSTD DESIGNS     54415
JUMBO BAG RED RETROSPOT               46181
WHITE HANGING HEART T-LIGHT HOLDER    36725
Name: Quantity, dtype: int64

Top 5 Countries by Transactions:
 Country
United Kingdom    354321
Germany             9040
France              8341
EIRE                7236
Spain               2484
Name: count, dtype: int64

📈 Step 4: RFM Feature Engineering...

🧮 Step 5: Clustering Methodology & Evaluation...
Clusters k=2 | Silhouette Score: 0.4329 | Inertia: 6481.23
Clusters k=3 | Silhouette Score: 0.3365 | Inertia: 4867.85
Clusters k=4 | Silhouette Score: 0.3371 | Inertia: 3938.51
Clusters k=5 | Silhouette Score: 0.3161 | Inertia: 3295.98

Discovered Mathem

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Ensure the models/ directory exists and assets are loaded
try:
    df = pd.read_csv('online_retail.csv')
    with open('models/scaler.pkl', 'rb') as f:
        scaler = pickle.load(f)
    with open('models/kmeans.pkl', 'rb') as f:
        best_kmeans = pickle.load(f)
    with open('models/cluster_mapping.pkl', 'rb') as f:
        cluster_mapping = pickle.load(f)
except FileNotFoundError:
    print("Please make sure online_retail.csv exists and build_models.py has been run first!")
    exit()

# Reconstruct Clean Data and RFM Base for Plotting
df = df.dropna(subset=['CustomerID', 'Description'])
df['CustomerID'] = df['CustomerID'].astype(int)
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['TotalAmount'] = df['Quantity'] * df['UnitPrice']

snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalAmount': 'sum'
}).reset_index()
rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

# Predict segments again for visual mapping
X_scaled = scaler.transform(np.log1p(rfm[['Recency', 'Frequency', 'Monetary']]))
rfm['Cluster'] = best_kmeans.predict(X_scaled)
rfm['Segment'] = rfm['Cluster'].map(cluster_mapping)

# Create a clean directory for documentation plots
import os
os.makedirs('plots', exist_ok=True)
sns.set_theme(style="whitegrid")

# -------------------------------------------------------------------------
# VISUALIZATION 1: Transactions by Country (Excluding UK for better scale)
# -------------------------------------------------------------------------
plt.figure(figsize=(10, 5))
country_data = df[df['Country'] != 'United Kingdom']['Country'].value_counts().head(10)
sns.barplot(x=country_data.values, y=country_data.index, palette='viridis')
plt.title('Top 10 International Markets by Transaction Volume (Excluding UK)')
plt.xlabel('Number of Transactions')
plt.tight_layout()
plt.savefig('plots/transactions_by_country.png', dpi=300)
plt.close()

# -------------------------------------------------------------------------
# VISUALIZATION 2: Purchase Trends Over Time (Monthly Revenue)
# -------------------------------------------------------------------------
plt.figure(figsize=(12, 5))
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M')
monthly_revenue = df.groupby('InvoiceMonth')['TotalAmount'].sum()
monthly_revenue.index = monthly_revenue.index.to_timestamp()
plt.plot(monthly_revenue.index, monthly_revenue.values, marker='o', color='#2b5c8f', linewidth=2.5)
plt.title('Global Monthly Revenue Trends (Historical Timeline Overview)')
plt.xlabel('Timeline')
plt.ylabel('Total Revenue Over Time ($)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('plots/purchase_trends_over_time.png', dpi=300)
plt.close()

# -------------------------------------------------------------------------
# VISUALIZATION 3: RFM Value Skew vs Log Transformation Distributions
# -------------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(np.log1p(rfm['Recency']), ax=axes[0], kde=True, color='crimson')
axes[0].set_title('Log-Transformed Recency Distribution')
sns.histplot(np.log1p(rfm['Frequency']), ax=axes[1], kde=True, color='teal')
axes[1].set_title('Log-Transformed Frequency Distribution')
sns.histplot(np.log1p(rfm['Monetary']), ax=axes[2], kde=True, color='gold')
axes[2].set_title('Log-Transformed Monetary Distribution')
plt.tight_layout()
plt.savefig('plots/rfm_distributions.png', dpi=300)
plt.close()

# -------------------------------------------------------------------------
# VISUALIZATION 4: Elbow Optimization Curve
# -------------------------------------------------------------------------
plt.figure(figsize=(7, 4))
inertia = []
k_range = range(1, 8)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertia.append(km.inertia_)
plt.plot(k_range, inertia, marker='D', color='purple')
plt.title('The Elbow Curve for Optimal Cluster Discovery')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia / Within-Cluster Sum of Squares')
plt.axvline(x=4, color='red', linestyle='--', label='Elbow Point (k=4)')
plt.legend()
plt.tight_layout()
plt.savefig('plots/elbow_curve.png', dpi=300)
plt.close()

# -------------------------------------------------------------------------
# VISUALIZATION 5: Final 2D Customer Segmentation Profiles (Scatter Plot)
# -------------------------------------------------------------------------
plt.figure(figsize=(10, 6))
# Using a log-scale representation to cleanly separate clusters visually
sns.scatterplot(
    data=rfm, 
    x='Frequency', 
    y='Monetary', 
    hue='Segment', 
    palette={'High-Value': 'green', 'Regular': 'blue', 'Occasional': 'orange', 'At-Risk': 'red'},
    alpha=0.7, 
    edgecolor='none'
)
plt.xscale('log')
plt.yscale('log')
plt.title('Customer Profiles Segment Scatter Matrix (Frequency vs Monetary Spend)')
plt.xlabel('Frequency Log Scale (Log of Total Purchase Counts)')
plt.ylabel('Monetary Log Scale (Log of Financial Outlay Value)')
plt.legend(title='Assigned Segment Tier')
plt.tight_layout()
plt.savefig('plots/customer_cluster_profiles.png', dpi=300)
plt.close()

print("🏁 All Step 3 Visualizations generated and saved inside your 'plots/' folder!")

C:\Users\kartik\AppData\Local\Temp\ipykernel_13852\3837675518.py:53: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=country_data.values, y=country_data.index, palette='viridis')


🏁 All Step 3 Visualizations generated and saved inside your 'plots/' folder!
